## Running Ironhack notebooks in VS Code

### One-off setup (already done)

Your `.venv_airflow` environment already has `ipykernel` and `pandas` installed, so you don't need to repeat this. It's here in case you ever make a new environment or need a new package.

1. In the VS Code terminal, make sure you're in `~/data_course`.
2. Switch the environment on: `source .venv_airflow/bin/activate` (the prompt should start with `(.venv_airflow)`).
3. Install what notebooks need: `pip install ipykernel pandas`
4. For any other package a lab needs later (for example numpy), use the same pattern: `pip install <package name>`

### Every time you add a new notebook

Each new .ipynb file needs its kernel picking once. After that, VS Code remembers it for that file.

1. **Drag it in.** Drag the downloaded .ipynb from Windows into your `01_notebooks` folder in the VS Code Explorer (left-hand panel). Rename it if you like (right-click → Rename), for example replacing DATE with the date.
2. **Open it.** Click the file in the Explorer.
3. **Click Select Kernel.** It's in the top-right corner of the notebook. (If it already says `.venv_airflow`, skip to step 7.)
4. **Click Python Environments…**
5. **Pick .venv_airflow.** Choose the line ending `.venv_airflow/bin/python`. Ignore the one marked "Recommended" (`/usr/bin/python3`): that's Ubuntu's own Python and it won't work.
6. **Check the top-right.** It should now say `.venv_airflow`.
7. **Run the import cell.** Click ▷ next to the import cell. A green tick and [1] mean it's working.

## Part 1 — YARN Components and Application Flow

### Q. 1 — Match YARN Components

Complete the table.

| Component | Responsibility |
|---|---|
| ResourceManager | |
| NodeManager | |
| ApplicationMaster | |
| Container | |

========================================================================
ANSWER Q1
========================================================================

Component: ResourceManager

Responsibility: To read data from the ApplicationMaster and from the NodeManagers, which read data from the individual machines, and then use this data to assign CPU and memory to each application's tasks. 'Manages cluster-wide resource allocation'.

====================================

Component: NodeManager

Responsibility: Runs on each worker machine. It launches and monitors containers on that machine and reports its resources back to the ResourceManager. 'Manages resources and containers on one worker'.

====================================

Component: ApplicationMaster

Responsibility: One per application. It asks the ResourceManager for containers and coordinates running that application's tasks. 'Coordinates one application'.

====================================

Component: Container

Responsibility: A slice of resources (memory and CPU) on one worker machine, where a single task actually runs. 'Provides allocated CPU and memory for work'.

========================================================================

### Q. 2 — Put the YARN Flow in Order

Arrange these steps:

```
Tasks run in containers
User submits an application
ApplicationMaster requests containers
ResourceManager starts an ApplicationMaster
ApplicationMaster reports completion
```

========================================================================
ANSWER Q2
========================================================================

User submits an application -->

ResourceManager starts an ApplicationMaster -->

ApplicationMaster requests containers --> 

Tasks run in containers -->

ApplicationMaster reports completion.

========================================================================

### Q. 3 — Allocate Simple Cluster Resources

A worker has:

```
Memory = 16 GB
CPU cores = 8
```

Each task container needs:

```
Memory = 4 GB
CPU cores = 2
```

Ignoring other system usage, what is the maximum number of such containers that can fit at the same time?

In [ ]:
worker_memory = 16
worker_cpu_cores = 8
container_memory = 4
container_cpu_cores = 2

def q3():
    if ((worker_memory) // (container_memory)) >= ((worker_cpu_cores) / (container_cpu_cores)):
       q3_answer = ((worker_cpu_cores) // (container_cpu_cores))
    else:
        q3_answer = ((worker_memory) // (container_memory)) 
    return(q3_answer)

answer = q3()

print(f"The maximum number of containers that can fit at the same time is {answer}.")
# '//' gets rid of the decimal. 


The maximum number of containers that can fit at the same time is 4.


### Q. 4 — MCQ: ResourceManager

Which YARN component makes cluster-wide resource decisions?

A. ResourceManager

B. NodeManager

C. Reducer

D. DataNode

========================================================================
ANSWER Q4
========================================================================

A. ResourceManager. (FYI: A NodeManager runs on each worker machine and looks after that one machine.)

========================================================================

## Part 2 — Map, Partition, Shuffle, Sort, and Reduce

### Q. 5 — Choose the Mapper Key

BritMart wants total sales by city.

Input columns:

```
order_id
city
amount_gbp
```

Which mapper output is suitable?

```
A. (city, amount_gbp)
B. (order_id, city)
C. (amount_gbp, order_id)
```

Explain why.

========================================================================
ANSWER Q5
========================================================================

A. (city, amount_gbp)

order_id is unique so would result in all the data getting sent across the network; costly and longer to process. The other two options also just don't make sense, they don't provide the answer.

========================================================================

### Q. 6 — Simulate Map and Reduce for Sales

Use:

```python
sales = [
    ("London", 20),
    ("Manchester", 30),
    ("London", 40),
    ("Leeds", 15),
]
```

Write Python code that groups by city and calculates total sales.

In [12]:
import pandas as pd
sales = [("London", 20), ("Manchester", 30), ("London", 40), ("Leeds", 15)]
sales_df = pd.DataFrame(sales, columns=["city", "amount"])

total_sales_by_city = sales_df.groupby("city", as_index = False)["amount"].sum().sort_values("amount", ascending = False)
total_sales_by_city

# OR

sales = [("London", 20), ("Manchester", 30), ("London", 40), ("Leeds", 15)]

total_sales_by_city_two = {}

for city, amount in sales:
    if city in total_sales_by_city_two:
        total_sales_by_city_two[city] += amount
    else:
        total_sales_by_city_two[city] = amount

print(total_sales_by_city_two)

# How the else part adds the city:
# Setting a value on a dictionary key works in two ways.
# If the key already exists, its value is replaced.
# If the key doesn't exist yet, Python creates it there and then.
# So no append is needed: putting a value into a new key automatically adds the city.
# Assigning to a key that doesn't exist yet creates it, so this adds the new city to the dictionary without needing append

# OR

# Keeping as a list not a dictionary is harder:
# sales = [("London", 20), ("Manchester", 30), ("London", 40), ("Leeds", 15)]
# total_sales_by_city_two = []
# for city, amount in sales:
#    cities_so_far = [row[0] for row in total_sales_by_city_two]
#    if city in cities_so_far:
#        position = cities_so_far.index(city)
#        total_sales_by_city_two[position] = (city, total_sales_by_city_two[position][1] + amount)
#    else:
#        total_sales_by_city_two.append((city, amount))
# print(total_sales_by_city_two)




{'London': 60, 'Manchester': 30, 'Leeds': 15}


### Q. 7 — Trace Shuffle and Sort

Mapper output is:

```
(London, 20)
(Leeds, 15)
(London, 40)
(Manchester, 30)
(Leeds, 10)
```

Show the grouped reducer input after shuffle and sort.

========================================================================
ANSWER Q7
========================================================================

London -> [20, 40]
Manchester -> [30]
Leeds -> [15, 10]

========================================================================

### Q. 8 — MCQ: Shuffle

What happens during shuffle?

A. The NameNode calculates business totals

B. HDFS deletes mapper input

C. Reducers create YARN queues

D. Mapper output is moved to reducers based on its key

========================================================================
ANSWER Q8
========================================================================

D. Mapper output is moved to reducers based on its key

========================================================================

## Part 3 — Combiners and Reducers

### Q. 9 — Decide Whether a Combiner Is Safe

Complete the table.

| Operation | Safe for a combiner? |
|---|---|
| Sum | |
| Maximum | |
| Word count | |
| Average calculated from partial averages only | |

========================================================================
ANSWER Q9
========================================================================

Operation: Sum

Safe for a combiner?: Yes.

====================================

Operation: Maximum

Safe for a combiner?: Yes.

====================================

Operation: Word count

Safe for a combiner?: Yes.

====================================

Operation: Average calculated from partial averages only

Safe for a combiner?: No. Mathematically, averages would break here. 

========================================================================

### Q. 10 — Simulate a Local Combiner

Mapper output:

```python
mapped = [
    ("London", 10),
    ("London", 20),
    ("Manchester", 15),
    ("London", 5),
]
```

Create a local combined result before the final reducer stage.

In [28]:
mapped = [("London", 10), ("London", 20), ("Manchester", 15), ("London", 5)]

import pandas as pd
mapped_df = pd.DataFrame(mapped, columns=["city", "amount"])
local_results = mapped_df.groupby("city", as_index = False)["amount"].sum()
local_results

# OR

mapped = [("London", 10), ("London", 20), ("Manchester", 15), ("London", 5)]

local_results = {}
for city, amount in mapped:
    if city in local_results:
        local_results[city] += amount
    else:
        local_results[city] = amount
# print(local_results)

# 'Take the quotes off "city" on lines 15 and 17. With quotes, Python uses the literal word "city" as the key instead of the actual city name. 
#  Your code would run, but you'd get {'city': 5}: every row gets saved under the one key "city", and each new one overwrites the last.'

# Ironhacks Answer:

from collections import defaultdict
 
mapped = [
    ("London", 10),
    ("London", 20),
    ("Manchester", 15),
    ("London", 5),
]
 
local_totals = defaultdict(int)
 
for key, value in mapped:
    local_totals[key] += value
 
combined = list(local_totals.items())
 
print(combined)

# The difference is that Ironhack's returns a list and mine returns a dictionary.
# Ironhack's uses defaultdict(int), which starts any new key at 0, so it doesn't need my if/else.
# Ironhack's fits the idea slightly better: a combiner's output carries on through the MapReduce flow,
# so it should look like what the mapper produced, a list of (key, value) pairs.
# The shuffle then sends those pairs to the reducers.

[('London', 35), ('Manchester', 15)]


### Q. 11 — Compare Combiner and Reducer

Complete:

| Area | Combiner | Reducer |
|---|---|---|
| Main purpose | | |
| Guaranteed to run? | | |
| Final output? | | |

========================================================================
ANSWER Q11
========================================================================

Area: Main purpose

Combiner: Does local aggregation on each mapper's output before the shuffle, to cut down the data sent across the network. 'Reduce intermediate data locally'.

Reducer: Does the final aggregation on all the grouped values for each key after the shuffle. 'Produce the final result for grouped keys'.

====================================

Area: Guaranteed to run?

Combiner: No. It's an optional optimisation that Hadoop may run zero, one or several times.

Reducer: Yes, whenever the job has a reduce stage.

====================================

Area: Final output?

Combiner: No. Its output goes on to the shuffle and then the reducer.

Reducer: Yes. Its output is the final job result.

========================================================================

### Q. 12 — MCQ: Combiner

What is the main benefit of a combiner?

A. It replaces the ResourceManager

B. It can reduce intermediate data before shuffle

C. It stores HDFS metadata

D. It creates SQL tables automatically

========================================================================
ANSWER Q12
========================================================================

B. It can reduce intermediate data before shuffle.

========================================================================

## Part 4 — Hive, Impala, Spark, and Tool Choice

### Q. 13 — Compare the Tools

Complete:

| Tool | Main strength |
|---|---|
| Hive | |
| Impala | |
| Spark | |
| Classic MapReduce | |

========================================================================
ANSWER Q13
========================================================================

Tool: Hive

Main strength: 'SQL-based batch analytics.' I'm assuming this is to do with how the allocation manager allocates task across the machines, obviously python and sql etc do things differently, so different allocations and grouping of tasks makes sense.

Correction: Batch analytics isn't about how tasks are allocated, and it isn't about Python vs SQL (Hive is SQL only). It means Hive turns a SQL query into batch jobs (originally MapReduce, now often Tez or Spark) that run across the machines. That makes it good for large scheduled queries like nightly reports, but slower than Impala for quick interactive queries.

====================================

Tool: Impala

Main strength: 'Interactive low-latency SQL'.

====================================

Tool: Spark

Main strength: 'Flexible multi-stage processing using APIs such as Python'.

====================================

Tool: Classic MapReduce

Main strength: 'Stable low-level batch-processing model used in older Hadoop workloads'.

========================================================================

### Q. 14 — Choose the Right Tool

Choose **Hive**, **Impala**, **Spark**, or **classic MapReduce**.

```
1. Nightly SQL aggregation.
2. Analyst wants fast ad hoc SQL.
3. Multi-stage Python transformation.
4. Existing stable Java MapReduce job needs maintenance.
```

========================================================================
ANSWER Q14
========================================================================

Note: Impala (low latency): you run a query and get results in seconds, like querying a normal database.

1. Nightly SQL aggregation. Hive.


====================================

2. Analyst wants fast ad hoc SQL. Impala.


====================================

3. Multi-stage Python transformation. Spark. 


====================================

4. Existing stable Java MapReduce job needs maintenance. Classic Map Reduce. 


========================================================================

### Q. 15 — Explain Why Classic MapReduce Can Be Slow

BritMart has a five-stage processing workflow.

Explain why repeatedly writing intermediate data to disk can make classic MapReduce slower than engines that can reuse data more efficiently between stages.

========================================================================
ANSWER Q15
========================================================================

'Classic MapReduce commonly writes intermediate stage output to storage before the next stage reads it. For multi-stage or iterative work, repeated disk I/O can add significant overhead. Engines such as Spark can often keep or reuse intermediate data more efficiently.'

========================================================================

### Q. 16 — MCQ: Interactive SQL

Which tool is most suitable for interactive SQL queries on Hadoop data?

A. ResourceManager

B. MapReduce Partitioner

C. Impala

D. NodeManager

========================================================================
ANSWER Q16
========================================================================

C. Impala.

Impala is a SQL tool built for interactive, low-latency queries. It keeps its query engine running on the cluster all the time, so there is no slow start-up and results come back in seconds, which means you can ask follow-up questions straight away. 

========================================================================

## Optional Practice

### Q. 17 — Simulate MapReduce with Three Input Batches

Use:

```python
batch_1 = [
    ("London", 10),
    ("Leeds", 20),
]

batch_2 = [
    ("London", 30),
    ("Manchester", 15),
]

batch_3 = [
    ("Leeds", 25),
    ("London", 5),
]
```

Combine the batches and calculate final totals by city.

In [ ]:
import pandas as pd
batch_1 = [("London", 10), ("Leeds", 20)]
batch_2 = [("London", 30), ("Manchester", 15)]
batch_3 = [("Leeds", 25), ("London", 5)]

batch_1 = pd.DataFrame(batch_1, columns=["city", "amount"])
batch_2 = pd.DataFrame(batch_2, columns=["city", "amount"])
batch_3 = pd.DataFrame(batch_3, columns=["city", "amount"])

combined = pd.concat([batch_1, batch_2, batch_3], ignore_index=True)
combined

city_totals = combined.groupby("city", as_index = False)["amount"].sum().sort_values("amount", ascending = False)
city_totals
# Without as_index=False, the result is a Series (no column names), so sort_values can't take "amount" and you'd need sort_values(ascending=False)

# Ironhacks Answer:

batch_1 = [("London", 10), ("Leeds", 20)]
batch_2 = [("London", 30), ("Manchester", 15)]
batch_3 = [("Leeds", 25), ("London", 5)]

all_rows = (batch_1 + batch_2 + batch_3)
 
grouped = defaultdict(list)
 
for city, amount in all_rows:
    grouped[city].append(amount)
 
result = {
    city: sum(values)
    for city, values in grouped.items()
}
 
print(result)

,city,amount
0,Leeds,45
1,London,45
2,Manchester,15


### Q. 18 — Find a Bad Mapper Key

Requirement:

```
total sales by city
```

A mapper emits:

```
(order_id, amount_gbp)
```

Explain why this key does not directly support the required grouping.

========================================================================
ANSWER Q18
========================================================================

Because you've used order_id as the key. This is a unique value and doesn't pre-aggregate by city. It's a poor key choice. Also, city isn't in the mapper output at all, so the reducer has no way to group the amounts by city.

========================================================================

### Q. 19 — Check Combiner Correctness

Two mappers contain:

```
Mapper 1 values: 10, 20
Mapper 2 values: 100
```

If each mapper sends only its local average:

```
15
100
```

and the reducer averages those two values, what result is produced?

What is the correct overall average?

In [ ]:
mapper_1_values = [10, 20]
mapper_2_values = [100]

total_mapper_1 = 0
count_mapper_1 = 0
total_mapper_2 = 0
count_mapper_2 = 0

for value in mapper_1_values:
    total_mapper_1 += value
    count_mapper_1 += 1
mapper_1_average = total_mapper_1 / count_mapper_1

for value in mapper_2_values:
    total_mapper_2 += value
    count_mapper_2 += 1
mapper_2_average = total_mapper_2 / count_mapper_2

total = 0
count = 0

list = [mapper_1_average, mapper_2_average]

for map in list:
    total += map
    count += 1
average = total / count

print(average)

# Should pick better names next time as they are python functions. 


57.5


### Q. 20 — Design a Tool Migration Decision

BritMart has an old MapReduce pipeline with:

```
join
filter
aggregation
three processing stages
```

The team wants Python and easier maintenance.

Choose a replacement from today's tools and explain:

```
why it fits
one benefit
one trade-off
whether HDFS data can remain in place
```

========================================================================
ANSWER Q20
========================================================================

Spark. 

Replacement: Spark

Why it fits: Spark can do the join, filter and aggregation in one job, and the team can write it in Python using PySpark (with DataFrames or Spark SQL), which makes it much easier to read and maintain than old Java MapReduce code.

One benefit: Spark keeps data in memory between stages, instead of writing to disk after each one like MapReduce, so a three-stage pipeline runs much faster.

One trade-off: Spark needs more memory, so the cluster may need tuning or more resources. The team also has to rewrite the old pipeline and test that it gives the same results.

Whether HDFS data can remain in place: Yes. Spark can read from and write to HDFS directly, and it can run on YARN, so the existing data and cluster can stay as they are.

========================================================================